# Value Iteration

You are on a [frozen lake](https://gymnasium.farama.org/environments/toy_text/frozen_lake/) of Ontario trying to find a treasure.
The frozen lake is slippery and you can slide from one location to another.
What's worse is that Lake Ontario is so large that there are areas where the ice is fragile---you can fall into the hole.
Luckily, being a good swimmer you can always get out of it but clearly you will suffer.
Upon reaching the treasure, you get a wish---if you further stay at the treasure for a longer period of time you get more wishes.

You are very well prepared so you know exactly where the holes are and where the treasure is.
You also know how likely you will slip.

### State space
```
MAP = [
    "FFFFFFFF",
    "FFFFFFFF",
    "FFGHFFFF",
    "FFFFFHFF",
    "FFFHFFFF",
    "FHHFFFHF",
    "FHFFHFHF",
    "FFFHFSFF",
]
```

The tile letters denote
- “S” for Start tile
- “G” for Goal tile
- “F” for frozen tile
- “H” for a tile with a hole

### Action space
```
LEFT = 0
DOWN = 1
RIGHT = 2
UP = 3
STAY = 4
```

### Transition
The player will move in intended direction with probability of 1/3 else will move in either perpendicular direction with equal probability of 1/3 in both directions.
When the player intends to stay, it stays in the current cell with probability of 1.

### Reward
The player receives +1 reward on staying at goal tile, -1 reward on staying at a hole tile, and +0 otherwise.

In [50]:
from grid_world import GridWorld

import numpy as np

In [51]:
env = GridWorld()

`transition` is a dictionary of dictionary, e.g.:
```
{
    ...
    59: {
        0: [
            (0.3333333333333333, 51, -1.0, False),
            (0.3333333333333333, 58, -1.0, False),
            (0.3333333333333333, 59, -1.0, False)
        ],
        1: [
            (0.3333333333333333, 58, -1.0, False),
            (0.3333333333333333, 59, -1.0, False),
            (0.3333333333333333, 60, -1.0, False)
        ],
        2: [
            (0.3333333333333333, 59, -1.0, False),
            (0.3333333333333333, 60, -1.0, False),
            (0.3333333333333333, 51, -1.0, False)
        ],
        3: [
            (0.3333333333333333, 60, -1.0, False),
            (0.3333333333333333, 51, -1.0, False),
            (0.3333333333333333, 58, -1.0, False)
        ],
        4: [
            (1.0, 59, -1.0, False)]
    },
    ...
}
```

- `59` corresponds to the 59'th cell, which can be represented as `(row_i, col_j)` by calling `get_cell(idx)`
- `0, 1, 2, 3, 4` are the actions
- each tuple corresponds to all possible transitions, where
  - index 0 means transition probability
  - index 1 means next state
  - index 2 means reward
  - index 3 means end of trajectory (Note this is always `False`)

In [52]:
transition = env.transition

In [53]:
map_action = {
    0: "L",
    1: "D",
    2: "R",
    3: "U",
    4: "S,"
}

def get_cell(idx):
    return (idx // env.nrow, idx % env.nrow)

def to_idx(row, col):
    return row * env.ncol + col

In [72]:
def greedy_policy(vf, gamma):
    policy = np.full(len(transition), -1)

    # ==================================================================
    # TODO: Implement greedy policy w.r.t. value function
 

    for ind in range(len(vf)):
        max_value = -float("inf")
        for action in transition[ind]:
            temp_value = 0
            for prob, next_state_ind, reward, _ in transition[ind][action]:
                temp_value += prob * (reward + gamma * vf[next_state_ind])
            if temp_value >= max_value:
                max_value = temp_value
                policy[ind] = action
    # ==================================================================

    return policy


In [73]:
def value_iteration(gamma: float, threshold: float, print_interval: int = 1):
    vf = np.zeros(len(transition))

    print("INITIAL VF")
    print(vf.reshape((env.nrow, env.ncol)))

    num_iterations = 0
    while True:
        num_iterations += 1

        vf_next = np.copy(vf)

        # ==================================================================
        # TODO: Implement value iteration
        for ind in range(len(vf)):
            for action in transition[ind]:
                temp_value = 0
                for prob, next_state_ind, reward, _ in transition[ind][action]:
                    temp_value += prob * (reward + gamma * vf[next_state_ind])
                vf_next[ind] = max(vf_next[ind], temp_value)

        if np.sum(vf_next-vf) < threshold:
            break

        # ==================================================================

        if num_iterations % print_interval == 0:
            print("ITER {} =========".format(num_iterations))
            print("Value function:")
            print(vf.reshape((env.nrow, env.ncol)))

            pi = greedy_policy(vf, gamma)
            print("Policy:")
            print(np.array([map_action[action] for action in pi]).reshape((env.nrow, env.ncol)))
        
        
        vf = np.copy(vf_next)

    return vf, num_iterations

In [68]:
gamma = 0.99
threshold = 1e-7

In [75]:
vf_star, num_iterations = value_iteration(gamma, threshold)

INITIAL VF
[[0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]]
ITER 1 =========
Value function:
[[0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]]
Policy:
[['S,' 'S,' 'S,' 'S,' 'S,' 'S,' 'S,' 'S,']
 ['S,' 'S,' 'S,' 'S,' 'S,' 'S,' 'S,' 'S,']
 ['S,' 'S,' 'S,' 'S,' 'S,' 'S,' 'S,' 'S,']
 ['S,' 'S,' 'S,' 'S,' 'S,' 'S,' 'S,' 'S,']
 ['S,' 'S,' 'S,' 'S,' 'S,' 'S,' 'S,' 'S,']
 ['S,' 'S,' 'S,' 'S,' 'S,' 'S,' 'S,' 'S,']
 ['S,' 'S,' 'S,' 'S,' 'S,' 'S,' 'S,' 'S,']
 ['S,' 'S,' 'S,' 'S,' 'S,' 'S,' 'S,' 'S,']]
ITER 2 =========
Value function:
[[0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 

In [60]:
print(vf_star.reshape((env.nrow, env.ncol)))

[[86.61057951 87.89303795 88.6209688  87.51625014 85.422706   82.7975296
  80.04027726 77.68615145]
 [87.9526841  89.8285325  92.03438356 89.06353906 85.91772872 82.68136925
  79.70848787 77.5227328 ]
 [88.74206855 92.22060659 99.99999984 90.3388786  85.87050864 81.83434547
  78.81922583 77.00424827]
 [87.8538411  89.62785128 91.52510163 87.7209417  83.95705524 79.43098718
  77.30330553 76.00222799]
 [85.77067568 86.28662164 86.07608407 83.02489513 80.82386857 77.93886348
  75.92351844 74.82909898]
 [82.22187253 81.16464128 80.47526731 79.65402406 77.87566806 75.50898028
  73.00056023 72.81162319]
 [78.49186858 77.14040611 77.13259956 76.11947678 73.87845753 72.90927192
  70.43280993 70.55322825]
 [76.18328421 75.51763851 75.18593815 73.03102526 72.35840197 71.54974982
  69.93170853 69.19407334]]


In [61]:
pi_star = greedy_policy(vf_star, gamma)

In [62]:
np.array([map_action[action] for action in pi_star]).reshape((env.nrow, env.ncol))

array([['R', 'R', 'L', 'L', 'L', 'L', 'L', 'U'],
       ['D', 'D', 'D', 'L', 'L', 'L', 'L', 'U'],
       ['U', 'R', 'S,', 'L', 'L', 'L', 'L', 'U'],
       ['U', 'U', 'U', 'U', 'L', 'L', 'U', 'U'],
       ['U', 'U', 'U', 'U', 'U', 'U', 'U', 'U'],
       ['U', 'U', 'U', 'U', 'U', 'U', 'U', 'U'],
       ['U', 'U', 'U', 'U', 'U', 'L', 'U', 'U'],
       ['L', 'L', 'L', 'L', 'L', 'L', 'L', 'U']], dtype='<U2')

### Credits
Bryan Chan, Nov. 2024